# Exercise 3: Building a Multi-Omics Glue Data Catalog
## BINFX410 — Chapter 10

---

## Learning Objectives
- Explain the role of a data catalog in a genomics data lake
- Configure and run an AWS Glue Crawler over mixed bioinformatics file types
- Identify where automated schema discovery fails on VCF and MAF formats
- Manually correct a Glue table definition using boto3
- Run a cross-table Athena query joining clinical metadata to expression data

**Estimated time:** 60 minutes  
**Dataset:** TCGA open access (`s3://tcga-2-open`) — RNA-seq counts, clinical TSV, somatic MAF  
**AWS services:** AWS Glue, S3, Amazon Athena

## Background: Why Genomics Needs a Data Catalog

A large genomics lab might have S3 buckets containing:
- 500 GEO studies downloaded over 5 years, each with different column names
- TCGA expression matrices in HTSeq, RSEM, and Kallisto formats
- VCF files from GATK, DeepVariant, and Strelka — all with different INFO fields
- Clinical metadata in Excel, TSV, and JSON formats

Without a catalog, every analyst must manually explore S3 and guess at schemas. With a catalog (AWS Glue Data Catalog), every file is registered with its schema, location, and format. Athena reads the catalog to know how to parse each file.

### How Glue Crawlers Work

```
1. Crawler samples files in S3 (reads first N rows)
2. Infers schema: column names, data types, delimiters
3. Writes table definition to Glue Data Catalog
4. Athena reads catalog → knows how to parse file on query
```

**The catch:** Crawlers work well on standard CSV/JSON. They struggle with:
- VCF `##` multi-line headers
- MAF `#version` comment lines
- GCT format (GTEx expression files with 2-line header)
- BAM/CRAM (binary formats — not queryable by Athena at all)

This exercise teaches you both what crawlers do well AND how to fix them when they fail.

## Section 1: Setup

In [ ]:
!pip install boto3 awswrangler pandas --quiet

In [15]:
import boto3
import awswrangler as wr
import pandas as pd
import io
import time
import json
from botocore import UNSIGNED
from botocore.config import Config

STUDENT_ID = "sab032226"  # CHANGE THIS
AWS_REGION = "us-east-1"
BUCKET = f"binfx410-datalake-{STUDENT_ID}"
ATHENA_RESULTS = f"s3://binfx410-athena-results-{STUDENT_ID}/"

s3   = boto3.client('s3',   region_name=AWS_REGION)
glue = boto3.client('glue', region_name=AWS_REGION)
s3_unsigned = boto3.client('s3', region_name=AWS_REGION,
                            config=Config(signature_version=UNSIGNED))
session = boto3.Session(region_name=AWS_REGION)

# Derive account ID automatically — avoids the cross-account PassRole error
# that occurs when a hardcoded account ID doesn't match the active credentials.
ACCOUNT_ID = boto3.client('sts', region_name=AWS_REGION).get_caller_identity()['Account']
GLUE_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/binfx410-glue-role"

print(f"Account ID : {ACCOUNT_ID}")
print(f"Glue role  : {GLUE_ROLE_ARN}")
print("Setup complete.")

Account ID : 495163878159
Glue role  : arn:aws:iam::495163878159:role/binfx410-glue-role
Setup complete.


## Section 2: Staging Multi-Omics TCGA Data

We'll stage three different file types from TCGA open access, each with a different format, to illustrate the catalog's role in tracking heterogeneous data.

In [16]:
# First, explore what's in the TCGA open access bucket
response = s3_unsigned.list_objects_v2(
    Bucket='tcga-2-open',
    MaxKeys=30
)
print("Sample contents of s3://tcga-2-open/:")
for obj in response.get('Contents', [])[:15]:
    print(f"  {obj['Key'][:80]}  ({obj['Size']/1024:.0f} KB)")

Sample contents of s3://tcga-2-open/:
  000000f5-8270-43b0-9b05-59d1c90a61e1/5d9fb1f2-45d8-4af0-9c53-fa76162706f5.methyl  (12836 KB)
  0000093b-2b25-4781-9c21-7401eeb3ef88/TCGA-READ.72029f21-a40c-42f9-80ea-4c3d9d971  (3402 KB)
  00000aad-5723-4805-ad8a-57f9d95b3d2f/  (0 KB)
  000029f0-723e-46e1-a120-c1060d7ad55b/1acd12d4-c8f1-4b29-a41f-80444bf3e17c_analys  (4 KB)
  00002fe8-ec8e-4e0e-a174-35f039c15d06/6057825020_R01C01_Grn.idat  (7906 KB)
  00003412-f77d-4263-8405-a9ea95c849b8/5d452492-ab54-43f5-a50e-fbe882a2af8a_run.xm  (76 KB)
  0000387a-9d49-410a-bab8-436144bb1850/TCGA-THCA.e0e3bbc2-4d34-45c4-96fa-0a7c7e0f0  (3 KB)
  00005051-36c7-4850-9e2c-243be54077ea/  (0 KB)
  00005051-36c7-4850-9e2c-243be54077ea/TAZZA_p_TCGAb3_62_63_64_65_NSP_GenomeWideSN  (31 KB)
  0000513d-3e24-4672-b832-f1e530d523a1/a13934d5-4a79-4f03-aa87-505786ce6407_analys  (8 KB)
  0000576d-0298-42d9-afac-d31959513294/MSK_252152917399_S01_CGH-v4_10_27Aug08__GCN  (31296 KB)
  000057f5-df42-48ce-9fc0-2f0fd0230f77/  (0 KB)


In [17]:
# Since TCGA files are organized by UUID which changes, we'll create
# representative synthetic files that exactly match TCGA schemas

# FILE 1: RNA-seq HTSeq count matrix (TCGA format)
# Two-column: gene_id (Ensembl), raw_count
htseq_content = """ENSG00000000003\t842
ENSG00000000005\t0
ENSG00000000419\t520
ENSG00000000457\t323
ENSG00000000460\t77
ENSG00000000938\t0
ENSG00000001036\t1947
ENSG00000001084\t723
ENSG00000001167\t455
ENSG00000001460\t234
__no_feature\t45231
__ambiguous\t3421
__too_low_aQual\t12
__not_aligned\t8932
__alignment_not_unique\t4521
"""

s3.put_object(
    Bucket=BUCKET,
    Key='bronze/expression/cancer_type=BRCA/sample=TCGA-A2-A0T2-01A/htseq_counts.txt',
    Body=htseq_content.encode()
)
print("Staged: HTSeq count file")

Staged: HTSeq count file


In [18]:
# FILE 2: Clinical metadata TSV (TCGA clinical format)
clinical_data = pd.DataFrame([
    {'case_id': 'TCGA-A2-A0T2', 'project_id': 'TCGA-BRCA', 'primary_diagnosis': 'Infiltrating duct carcinoma, NOS',
     'tumor_stage': 'stage iia', 'age_at_diagnosis': 18628, 'gender': 'female',
     'vital_status': 'Alive', 'days_to_death': None, 'days_to_last_follow_up': 3402,
     'race': 'white', 'ethnicity': 'not hispanic or latino', 'morphology': '8500/3'},
    {'case_id': 'TCGA-BH-A0B3', 'project_id': 'TCGA-BRCA', 'primary_diagnosis': 'Infiltrating duct carcinoma, NOS',
     'tumor_stage': 'stage iiia', 'age_at_diagnosis': 20089, 'gender': 'female',
     'vital_status': 'Dead', 'days_to_death': 731, 'days_to_last_follow_up': 731,
     'race': 'black or african american', 'ethnicity': 'not hispanic or latino', 'morphology': '8500/3'},
    {'case_id': 'TCGA-A8-A08O', 'project_id': 'TCGA-BRCA', 'primary_diagnosis': 'Lobular carcinoma, NOS',
     'tumor_stage': 'stage i', 'age_at_diagnosis': 23741, 'gender': 'female',
     'vital_status': 'Alive', 'days_to_death': None, 'days_to_last_follow_up': 1825,
     'race': 'white', 'ethnicity': 'not hispanic or latino', 'morphology': '8520/3'},
])

s3.put_object(
    Bucket=BUCKET,
    Key='bronze/clinical/cancer_type=BRCA/clinical.tsv',
    Body=clinical_data.to_csv(sep='\t', index=False).encode()
)
print("Staged: Clinical metadata TSV")

Staged: Clinical metadata TSV


In [19]:
# FILE 3: Somatic mutation MAF (Mutation Annotation Format)
# MAF has a #version comment line and a non-standard header — this will trip up the crawler
maf_content = """#version 2.4
Hugo_Symbol\tEntrez_Gene_Id\tCenter\tNCBI_Build\tChromosome\tStart_Position\tEnd_Position\tStrand\tVariant_Classification\tVariant_Type\tReference_Allele\tTumor_Seq_Allele1\tTumor_Seq_Allele2\tTumor_Sample_Barcode\tHGVSc\tHGVSp_Short\tt_depth\tt_ref_count\tt_alt_count
TP53\t7157\tBroad\tGRCh38\tchr17\t7675088\t7675088\t+\tMissense_Mutation\tSNP\tG\tG\tA\tTCGA-A2-A0T2-01A\tc.817C>T\tp.R273C\t128\t64\t64
PIK3CA\t5290\tBroad\tGRCh38\tchr3\t179234297\t179234297\t+\tMissense_Mutation\tSNP\tA\tA\tG\tTCGA-A2-A0T2-01A\tc.1633G>A\tp.E545K\t95\t45\t50
GATA3\t2625\tBroad\tGRCh38\tchr10\t8073507\t8073507\t+\tFrame_Shift_Del\tDEL\tCTCCCC\tCTCCCC\t-\tTCGA-BH-A0B3-01A\tc.1082del\tp.K361fs\t67\t31\t36
CDH1\t999\tBroad\tGRCh38\tchr16\t68771194\t68771194\t+\tSplice_Site\tSNP\tG\tG\tA\tTCGA-BH-A0B3-01A\tc.1380+1G>A\tp.X460_splice\t82\t40\t42
MAP3K1\t4214\tBroad\tGRCh38\tchr5\t56148145\t56148148\t+\tFrame_Shift_Del\tDEL\tCATG\tCATG\t-\tTCGA-A8-A08O-01A\tc.1258_1261del\tp.S420fs\t55\t28\t27
"""

s3.put_object(
    Bucket=BUCKET,
    Key='bronze/mutations/cancer_type=BRCA/somatic_mutations.maf',
    Body=maf_content.encode()
)
print("Staged: Somatic MAF file")
print("\nAll three file types staged in bronze zone:")
print(f"  s3://{BUCKET}/bronze/expression/")
print(f"  s3://{BUCKET}/bronze/clinical/")
print(f"  s3://{BUCKET}/bronze/mutations/")

Staged: Somatic MAF file

All three file types staged in bronze zone:
  s3://binfx410-datalake-sab032226/bronze/expression/
  s3://binfx410-datalake-sab032226/bronze/clinical/
  s3://binfx410-datalake-sab032226/bronze/mutations/


## Section 3: Creating Glue Databases

In [20]:
# Create a Glue database for the bronze zone
for db_name in ['binfx410_bronze', 'binfx410_silver']:
    try:
        glue.create_database(DatabaseInput={
            'Name': db_name,
            'Description': f'BINFX410 data lake {db_name.split("_")[1]} zone'
        })
        print(f"Created database: {db_name}")
    except glue.exceptions.AlreadyExistsException:
        print(f"Database exists: {db_name}")

Database exists: binfx410_bronze
Database exists: binfx410_silver


## Section 4: Configuring and Running a Glue Crawler

In [21]:
# Ensure binfx410-glue-role exists and has the correct trust policy.
# The crawler will fail with InvalidInputException if the role's trust policy
# does not include glue.amazonaws.com as a trusted principal.

import json, time

iam = boto3.client('iam', region_name=AWS_REGION)
GLUE_ROLE_NAME = 'binfx410-glue-role'

TRUST_POLICY = json.dumps({
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": {"Service": "glue.amazonaws.com"},
        "Action": "sts:AssumeRole"
    }]
})

try:
    role = iam.get_role(RoleName=GLUE_ROLE_NAME)
    # Role exists — verify glue.amazonaws.com is a trusted principal
    trust = role['Role']['AssumeRolePolicyDocument']
    principals = str([s.get('Principal', {}) for s in trust.get('Statement', [])])
    if 'glue.amazonaws.com' not in principals:
        iam.update_assume_role_policy(
            RoleName=GLUE_ROLE_NAME,
            PolicyDocument=TRUST_POLICY
        )
        print(f"Fixed trust policy for {GLUE_ROLE_NAME} — added glue.amazonaws.com")
        time.sleep(10)  # allow IAM to propagate
    else:
        print(f"Role {GLUE_ROLE_NAME} exists with correct trust policy.")

except iam.exceptions.NoSuchEntityException:
    # Role doesn't exist — create it from scratch
    iam.create_role(
        RoleName=GLUE_ROLE_NAME,
        AssumeRolePolicyDocument=TRUST_POLICY,
        Description='Glue service role for BINFX410 exercises'
    )
    # AWS managed policy grants Glue access to Glue APIs, CloudWatch, and S3 (general)
    iam.attach_role_policy(
        RoleName=GLUE_ROLE_NAME,
        PolicyArn='arn:aws:iam::aws:policy/service-role/AWSGlueServiceRole'
    )
    # Inline policy scoped to this student's S3 bucket
    iam.put_role_policy(
        RoleName=GLUE_ROLE_NAME,
        PolicyName='binfx410-s3-access',
        PolicyDocument=json.dumps({
            "Version": "2012-10-17",
            "Statement": [{
                "Effect": "Allow",
                "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject", "s3:ListBucket"],
                "Resource": [
                    f"arn:aws:s3:::{BUCKET}",
                    f"arn:aws:s3:::{BUCKET}/*"
                ]
            }]
        })
    )
    print(f"Created role: {GLUE_ROLE_NAME}")
    print(f"  Attached: AWSGlueServiceRole (managed)")
    print(f"  Attached: S3 access for s3://{BUCKET}")
    time.sleep(15)  # IAM changes take a moment to propagate

# Refresh GLUE_ROLE_ARN from the live role (avoids any stale ARN from setup cell)
GLUE_ROLE_ARN = iam.get_role(RoleName=GLUE_ROLE_NAME)['Role']['Arn']
print(f"\nGlue role ARN: {GLUE_ROLE_ARN}")

Created role: binfx410-glue-role
  Attached: AWSGlueServiceRole (managed)
  Attached: S3 access for s3://binfx410-datalake-sab032226

Glue role ARN: arn:aws:iam::495163878159:role/binfx410-glue-role


In [22]:
# Create a Glue Crawler to automatically discover schemas in our bronze zone
CRAWLER_NAME = f"binfx410-bronze-crawler-{STUDENT_ID}"

try:
    glue.create_crawler(
        Name=CRAWLER_NAME,
        Role=GLUE_ROLE_ARN,
        DatabaseName='binfx410_bronze',
        Description='Crawl BINFX410 bronze zone for multi-omics data',
        Targets={
            'S3Targets': [
                {'Path': f's3://{BUCKET}/bronze/expression/'},
                {'Path': f's3://{BUCKET}/bronze/clinical/'},
                {'Path': f's3://{BUCKET}/bronze/mutations/'},
            ]
        },
        SchemaChangePolicy={
            'UpdateBehavior': 'UPDATE_IN_DATABASE',
            'DeleteBehavior': 'DEPRECATE_IN_DATABASE'
        },
        RecrawlPolicy={'RecrawlBehavior': 'CRAWL_EVERYTHING'},
        TablePrefix='bronze_'
    )
    print(f"Created crawler: {CRAWLER_NAME}")
except glue.exceptions.AlreadyExistsException:
    print(f"Crawler already exists: {CRAWLER_NAME}")

Created crawler: binfx410-bronze-crawler-sab032226


In [23]:
# Start the crawler
glue.start_crawler(Name=CRAWLER_NAME)
print(f"Crawler started. Polling for completion...")

# Poll until complete
for i in range(30):  # max 5 minutes
    time.sleep(10)
    response = glue.get_crawler(Name=CRAWLER_NAME)
    state = response['Crawler']['State']
    print(f"  [{i*10}s] State: {state}")
    if state == 'READY':
        last_run = response['Crawler'].get('LastCrawl', {})
        print(f"  Crawler finished: {last_run.get('Status', 'unknown')}")
        print(f"  Tables created/updated: {last_run.get('TablesCreated', 0)} created, {last_run.get('TablesUpdated', 0)} updated")
        break
else:
    print("Crawler still running — check the Glue console.")

Crawler started. Polling for completion...
  [0s] State: RUNNING
  [10s] State: RUNNING
  [20s] State: RUNNING
  [30s] State: RUNNING
  [40s] State: RUNNING
  [50s] State: RUNNING
  [60s] State: STOPPING
  [70s] State: STOPPING
  [80s] State: STOPPING
  [90s] State: STOPPING
  [100s] State: STOPPING
  [110s] State: STOPPING
  [120s] State: STOPPING
  [130s] State: READY
  Crawler finished: SUCCEEDED
  Tables created/updated: 0 created, 0 updated


## Section 5: Inspecting Discovered Tables

In [24]:
# List tables discovered by the crawler
tables_response = glue.get_tables(DatabaseName='binfx410_bronze')

print(f"Tables in binfx410_bronze:")
for table in tables_response['TableList']:
    cols = table['StorageDescriptor']['Columns']
    print(f"\n  Table: {table['Name']}")
    print(f"  Location: {table['StorageDescriptor']['Location']}")
    print(f"  Format: {table['StorageDescriptor'].get('InputFormat', 'unknown')}")
    print(f"  Columns ({len(cols)}):")
    for col in cols[:5]:  # first 5 columns
        print(f"    {col['Name']}: {col['Type']}")
    if len(cols) > 5:
        print(f"    ... and {len(cols)-5} more")

Tables in binfx410_bronze:

  Table: bronze_clinical
  Location: s3://binfx410-datalake-sab032226/bronze/clinical/
  Format: org.apache.hadoop.mapred.TextInputFormat
  Columns (12):
    case_id: string
    project_id: string
    primary_diagnosis: string
    tumor_stage: string
    age_at_diagnosis: bigint
    ... and 7 more

  Table: bronze_expression
  Location: s3://binfx410-datalake-sab032226/bronze/expression/
  Format: org.apache.hadoop.mapred.TextInputFormat
  Columns (2):
    col0: string
    col1: bigint

  Table: bronze_mutations
  Location: s3://binfx410-datalake-sab032226/bronze/mutations/
  Format: org.apache.hadoop.mapred.TextInputFormat
  Columns (19):
    hugo_symbol: string
    entrez_gene_id: bigint
    center: string
    ncbi_build: string
    chromosome: string
    ... and 14 more

  Table: clinvar_parquet
  Location: s3://binfx410-datalake-sab032226/silver/variants/source=clinvar/
  Format: org.apache.hadoop.hive.ql.io.parquet.MapredParquetInputFormat
  Columns (43

## Section 6: Where the Crawler Fails — MAF Format

The MAF file has a `#version 2.4` comment line **before** the column header. The crawler reads this as the header, producing a completely wrong schema.

This is a common problem with bioinformatics formats — they were designed for human readability and tool compatibility, not machine-parseable schema discovery.

In [25]:
# Look at what the crawler produced for the MAF table
try:
    maf_table = glue.get_table(
        DatabaseName='binfx410_bronze',
        Name='bronze_mutations'  # Glue adds prefix from TablePrefix setting
    )
    cols = maf_table['Table']['StorageDescriptor']['Columns']
    print("Crawler-inferred MAF schema:")
    for col in cols:
        print(f"  {col['Name']}: {col['Type']}")
    print()
    print("Problem: The crawler used '#version 2.4' as the header!")
    print("The actual MAF header (Hugo_Symbol, Chromosome, etc.) was skipped.")
except Exception as e:
    print(f"Note: {e}")
    print("The crawler may have named the table differently. Check Glue console for actual table name.")

Crawler-inferred MAF schema:
  hugo_symbol: string
  entrez_gene_id: bigint
  center: string
  ncbi_build: string
  chromosome: string
  start_position: bigint
  end_position: bigint
  strand: string
  variant_classification: string
  variant_type: string
  reference_allele: string
  tumor_seq_allele1: string
  tumor_seq_allele2: string
  tumor_sample_barcode: string
  hgvsc: string
  hgvsp_short: string
  t_depth: bigint
  t_ref_count: bigint
  t_alt_count: bigint

Problem: The crawler used '#version 2.4' as the header!
The actual MAF header (Hugo_Symbol, Chromosome, etc.) was skipped.


## Section 7: Manually Fixing the MAF Table Definition

In [ ]:
# Manually create/update the MAF table with the correct schema
# The correct MAF 2.4 columns for the key fields we need

MAF_COLUMNS = [
    {'Name': 'hugo_symbol',           'Type': 'string'},
    {'Name': 'entrez_gene_id',        'Type': 'bigint'},
    {'Name': 'center',                'Type': 'string'},
    {'Name': 'ncbi_build',            'Type': 'string'},
    {'Name': 'chromosome',            'Type': 'string'},
    {'Name': 'start_position',        'Type': 'bigint'},
    {'Name': 'end_position',          'Type': 'bigint'},
    {'Name': 'strand',                'Type': 'string'},
    {'Name': 'variant_classification', 'Type': 'string'},
    {'Name': 'variant_type',          'Type': 'string'},
    {'Name': 'reference_allele',      'Type': 'string'},
    {'Name': 'tumor_seq_allele1',     'Type': 'string'},
    {'Name': 'tumor_seq_allele2',     'Type': 'string'},
    {'Name': 'tumor_sample_barcode',  'Type': 'string'},
    {'Name': 'hgvsc',                 'Type': 'string'},
    {'Name': 'hgvsp_short',           'Type': 'string'},
    {'Name': 't_depth',               'Type': 'int'},
    {'Name': 't_ref_count',           'Type': 'int'},
    {'Name': 't_alt_count',           'Type': 'int'},
]

try:
    # Delete the wrong crawler-created table if it exists
    glue.delete_table(DatabaseName='binfx410_bronze', Name='bronze_mutations')
    print("Deleted incorrect crawler-generated table.")
except:
    pass

# Create the correct table
glue.create_table(
    DatabaseName='binfx410_bronze',
    TableInput={
        'Name': 'somatic_mutations_maf',
        'Description': 'TCGA somatic mutations in MAF 2.4 format',
        'StorageDescriptor': {
            'Columns': MAF_COLUMNS,
            'Location': f's3://{BUCKET}/bronze/mutations/',
            'InputFormat': 'org.apache.hadoop.mapred.TextInputFormat',
            'OutputFormat': 'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat',
            'SerdeInfo': {
                'SerializationLibrary': 'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe',
                'Parameters': {
                    'field.delim': '\t',
                    'serialization.format': '\t'
                }
            }
        },
        'Parameters': {
            'skip.header.line.count': '2',  # skip #version line AND column header
            'classification': 'csv'
        },
        'PartitionKeys': [
            {'Name': 'cancer_type', 'Type': 'string'}
        ]
    }
)
print("Manually created correct somatic_mutations_maf table.")
print(f"  {len(MAF_COLUMNS)} columns defined correctly.")

In [ ]:
# Add the partition manually (since we created the table manually, not via crawler)
glue.create_partition(
    DatabaseName='binfx410_bronze',
    TableName='somatic_mutations_maf',
    PartitionInput={
        'Values': ['BRCA'],
        'StorageDescriptor': {
            'Columns': MAF_COLUMNS,
            'Location': f's3://{BUCKET}/bronze/mutations/cancer_type=BRCA/',
            'InputFormat': 'org.apache.hadoop.mapred.TextInputFormat',
            'OutputFormat': 'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat',
            'SerdeInfo': {
                'SerializationLibrary': 'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe',
                'Parameters': {'field.delim': '\t'}
            }
        }
    }
)
print("Partition 'cancer_type=BRCA' added.")

## Section 8: Querying Across Tables with Athena

In [ ]:
# First, manually register the clinical TSV as a Glue table
CLINICAL_COLUMNS = [
    {'Name': 'case_id', 'Type': 'string'},
    {'Name': 'project_id', 'Type': 'string'},
    {'Name': 'primary_diagnosis', 'Type': 'string'},
    {'Name': 'tumor_stage', 'Type': 'string'},
    {'Name': 'age_at_diagnosis', 'Type': 'int'},
    {'Name': 'gender', 'Type': 'string'},
    {'Name': 'vital_status', 'Type': 'string'},
    {'Name': 'days_to_death', 'Type': 'double'},
    {'Name': 'days_to_last_follow_up', 'Type': 'int'},
    {'Name': 'race', 'Type': 'string'},
    {'Name': 'ethnicity', 'Type': 'string'},
    {'Name': 'morphology', 'Type': 'string'},
]

try:
    glue.delete_table(DatabaseName='binfx410_bronze', Name='clinical_metadata')
except: pass

glue.create_table(
    DatabaseName='binfx410_bronze',
    TableInput={
        'Name': 'clinical_metadata',
        'StorageDescriptor': {
            'Columns': CLINICAL_COLUMNS,
            'Location': f's3://{BUCKET}/bronze/clinical/',
            'InputFormat': 'org.apache.hadoop.mapred.TextInputFormat',
            'OutputFormat': 'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat',
            'SerdeInfo': {
                'SerializationLibrary': 'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe',
                'Parameters': {'field.delim': '\t'}
            }
        },
        'Parameters': {'skip.header.line.count': '1', 'classification': 'csv'},
        'PartitionKeys': [{'Name': 'cancer_type', 'Type': 'string'}]
    }
)
glue.create_partition(
    DatabaseName='binfx410_bronze',
    TableName='clinical_metadata',
    PartitionInput={
        'Values': ['BRCA'],
        'StorageDescriptor': {
            'Columns': CLINICAL_COLUMNS,
            'Location': f's3://{BUCKET}/bronze/clinical/cancer_type=BRCA/',
            'InputFormat': 'org.apache.hadoop.mapred.TextInputFormat',
            'OutputFormat': 'org.apache.hadoop.hive.ql.io.HiveIgnoreKeyTextOutputFormat',
            'SerdeInfo': {'SerializationLibrary': 'org.apache.hadoop.hive.serde2.lazy.LazySimpleSerDe', 'Parameters': {'field.delim': '\t'}}
        }
    }
)
print("Clinical metadata table registered.")

In [ ]:
# Cross-table query: join mutations to clinical metadata
cross_table_query = """
SELECT
    c.case_id,
    c.tumor_stage,
    c.vital_status,
    c.age_at_diagnosis / 365.25 AS age_years,
    m.hugo_symbol,
    m.variant_classification,
    m.hgvsp_short
FROM binfx410_bronze.somatic_mutations_maf m
JOIN binfx410_bronze.clinical_metadata c
    ON m.tumor_sample_barcode LIKE CONCAT(c.case_id, '%')
WHERE m.cancer_type = 'BRCA'
  AND m.variant_classification IN ('Missense_Mutation', 'Frame_Shift_Del', 'Splice_Site')
ORDER BY c.case_id, m.hugo_symbol
"""

df_cross = wr.athena.read_sql_query(
    sql=cross_table_query,
    database='binfx410_bronze',
    s3_output=ATHENA_RESULTS,
    boto3_session=session
)

print("Cross-table query result (mutations joined to clinical data):")
df_cross

## Exercise: YOUR CODE HERE

In [ ]:
# TASK 1: Create a second sample for the expression data
# Write an HTSeq count file for sample TCGA-BH-A0B3 with different counts
# Stage it to: bronze/expression/cancer_type=BRCA/sample=TCGA-BH-A0B3/htseq_counts.txt

# YOUR CODE HERE

In [ ]:
# TASK 2: Write a Glue table definition for the HTSeq count files
# The HTSeq format is: two tab-separated columns: gene_id (string), raw_count (bigint)
# No header line (HTSeq does not write a header)
# Partition by: cancer_type, sample

# YOUR CODE HERE
htseq_columns = [
    # fill in column definitions
]

In [ ]:
# TASK 3: Write an Athena query that answers:
# "What is the raw count of ENSG00000001036 in each BRCA sample?"

# YOUR CODE HERE
gene_query = """
-- YOUR SQL HERE
"""

## Reflection Questions

*(Double-click to edit)*

1. **The Glue crawler failed on the MAF file** because of the `#version` comment line. What would you tell a bioinformatics tool developer about MAF file format design if they want their format to be cloud-queryable?

2. **A data catalog stores metadata, not data.** If you delete the Glue table definition for `somatic_mutations_maf`, what happens to the underlying MAF file in S3? What would happen if you re-ran the crawler?

3. **Schema drift is common in genomics.** TCGA MAF files from different analysis centers have slightly different columns. How would you handle a situation where 80% of your MAF files have 19 columns but 20% have 25 columns (with extra annotation fields)?

4. **The Glue Data Catalog is a regional service.** If your lab has sequencing data in S3 buckets in both us-east-1 and eu-west-1 (for GDPR compliance), how would you manage a unified catalog?

5. **Compare two approaches for organizing TCGA data in S3:**
   - Option A: One prefix per file type (`/expression/`, `/mutations/`, `/clinical/`)
   - Option B: One prefix per cancer type (`/BRCA/`, `/GBM/`, `/LUAD/`)
   
   Which would be better for a researcher who studies one cancer type? Which would be better for a pan-cancer analysis? How would your partitioning strategy reflect this choice?

*(Write answers here)*

1. 

2. 

3. 

4. 

5. 